# Project Delphi (Merlin)

## 01 - Data Preparation

### Project Overview:

Build a what-if machine to predict YouTube video performance *before* it's published. Instead of guessing, this system helps a creator make data-driven decisions -- testing potential titles, formats, and timing to maximize views and subscriber growth.

---

### The Pipeline:
1. **01_data_preparation**: Load, cleand and engineer base features
2. **02_title_embeddings**: Use Sentence Transformers (`all-MiniLM-L6-v2`) to embed video titles
3. **03_model_training_xgboost**: Train two XGBoost regressors to predict:
   - (a) total views
   - (b) total subscribers added
4. **04_stremlit_app**: Deploy as **Merlin**, an interactive web app
---

### This Notebook

Focus on preparing the dataset for modeling -- from loading and cleaning to feature engineering and saving a clean version for downstream use.

## Load (and clean) the data

Let's begin by uploading the dataset from YouTube. Then we'll get to work on preparing it for the model building stage.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Import pandas as load the CSV
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/project_delphi/data/project_delphi_data_raw.csv')
df.head()

,Content,Video title,Video publish time,Duration,Views,Subscribers
0,Total,NaN,NaN,NaN,741693.0,1808.0
1,nXjgpEBi054,49ers vs. Seahawks Livestream Watch Party w/ D...,"Sep 7, 2025",12564.0,12910.0,87.0
2,gRccD22-WcA,49ers vs. Bills Livestream Watch Party w/ Damon,"Dec 2, 2024",11467.0,11717.0,54.0
3,TUbMw5E7WWQ,Radio Layoffs Hit KNBR (Again) - Damon & Sam B...,"Nov 14, 2024",5306.0,10666.0,45.0
4,grBWI3qyGv8,49ers vs. Packers Watch Party Livestream w/ Damon,"Nov 25, 2024",13399.0,9780.0,72.0


There's an immediate problem. There are videos with 3 views... what?!

In [ ]:
# Check the minimum number of views for a video
df

,Content,Video title,Video publish time,Duration,Views,Subscribers
0,Total,NaN,NaN,NaN,741693.0,1808.0
1,nXjgpEBi054,49ers vs. Seahawks Livestream Watch Party w/ D...,"Sep 7, 2025",12564.0,12910.0,87.0
2,gRccD22-WcA,49ers vs. Bills Livestream Watch Party w/ Damon,"Dec 2, 2024",11467.0,11717.0,54.0
3,TUbMw5E7WWQ,Radio Layoffs Hit KNBR (Again) - Damon & Sam B...,"Nov 14, 2024",5306.0,10666.0,45.0
4,grBWI3qyGv8,49ers vs. Packers Watch Party Livestream w/ Damon,"Nov 25, 2024",13399.0,9780.0,72.0
...,...,...,...,...,...,...
497,9FHnN88_y6o,NFL's Wild Card Weekend,"Jan 13, 2024",3047.0,3.0,0.0
498,BHH2M7jXQXs,Trayce Jackson-Davis Has History On His Side,"Jun 26, 2023",4918.0,3.0,0.0
499,Czrk4OpmK7U,Super Bowl Week Wake Up w/ Damon & Larry,"Feb 5, 2024",5362.0,3.0,0.0
500,DA-72CWcj7w,Kyle Madson Joins Damon,"Oct 18, 2023",2035.0,3.0,0.0


Some of those videos with 3 views are beyond the 365-day timeframe that we want to train the model on. Let's clear them out.

In [ ]:
# Transform the dates to datetime
df['Video publish time'] = pd.to_datetime(df['Video publish time'], errors='coerce')

# Drop the 'Total' row (the NaN publish time)
df = df.dropna(subset=['Video publish time'])

# Define cutoff date (one year back from today)
cutoff_date = pd.Timestamp.today() - pd.Timedelta(days=365)

# Filter to only videos published in the last 365 days
df = df[df['Video publish time'] >= cutoff_date]

# Check the result
df[['Video publish time']].agg(['min', 'max'])
len(df)

336

In [ ]:
# One more visual check
df.head()

,Content,Video title,Video publish time,Duration,Views,Subscribers
1,nXjgpEBi054,49ers vs. Seahawks Livestream Watch Party w/ D...,2025-09-07,12564.0,12910.0,87.0
2,gRccD22-WcA,49ers vs. Bills Livestream Watch Party w/ Damon,2024-12-02,11467.0,11717.0,54.0
3,TUbMw5E7WWQ,Radio Layoffs Hit KNBR (Again) - Damon & Sam B...,2024-11-14,5306.0,10666.0,45.0
4,grBWI3qyGv8,49ers vs. Packers Watch Party Livestream w/ Damon,2024-11-25,13399.0,9780.0,72.0
5,I8lYR8HGl9o,49ers vs. Rams - Watch Party Livestream w/ Damon,2024-12-13,13090.0,9434.0,27.0


Time to do a bit more cleaning -- starting with the column names.

In [ ]:
# Lower case all the column names
df = df.rename(columns={
    'Content': 'video_id',
    'Video title': 'title',
    'Video publish time': 'publish_date',
    'Duration': 'duration_seconds',
    'Views': 'views_lifetime',
    'Subscribers': 'subs_lifetime'
})

# Check the first 3 results
df.head(3)

,video_id,title,publish_date,duration_seconds,views_lifetime,subs_lifetime
1,nXjgpEBi054,49ers vs. Seahawks Livestream Watch Party w/ D...,2025-09-07,12564.0,12910.0,87.0
2,gRccD22-WcA,49ers vs. Bills Livestream Watch Party w/ Damon,2024-12-02,11467.0,11717.0,54.0
3,TUbMw5E7WWQ,Radio Layoffs Hit KNBR (Again) - Damon & Sam B...,2024-11-14,5306.0,10666.0,45.0


One more issue to clean up: Need to clear out all the NFL watch party, or "livestream" shows. Instead, we just want to focus on the daily show, which generally clocks in at an hour or less. Let's see if we can find them all.

In [ ]:
# Display all videos that include the word "Livestream" in the title
df[df['title'].str.contains('Livestream', case=False, na=False)][['title', 'duration_seconds', 'views_lifetime']].sort_values('duration_seconds', ascending=False)

,title,duration_seconds,views_lifetime
22,49ers vs. Rams Livestream Watch Party w/ Damon...,14375.0,4902.0
7,49ers vs. Seahawks - Livestream Watch Party w/...,13666.0,8941.0
4,49ers vs. Packers Watch Party Livestream w/ Damon,13399.0,9780.0
12,49ers vs. Buccaneers Livestream Watch Party w/...,13127.0,6067.0
5,49ers vs. Rams - Watch Party Livestream w/ Damon,13090.0,9434.0
13,49ers vs. Cardinals Livestream Watch Party w/ ...,12704.0,6010.0
8,49ers vs. Jaguars Livestream Watch Party w/ Da...,12669.0,8845.0
1,49ers vs. Seahawks Livestream Watch Party w/ D...,12564.0,12910.0
6,49ers vs. Saints Watch Party Livestream w/ Dam...,12526.0,9271.0
15,49ers vs. Flacons Livestream Watch Party w/ Da...,12473.0,5754.0


Let's get rid of all those livestreams, including those outlier shows at the bottom.

In [ ]:
# Remove all Livestream / Watch Party episodes -- long or short
df = df[~df['title'].str.contains('Livestream|Watch Party', case=False, na=False)]

# Check to make sure the DataFrane is clean
df[df['title'].str.contains('Livestream|Watch Party', case=False, na=False)]

,video_id,title,publish_date,duration_seconds,views_lifetime,subs_lifetime


Perfect. Let's see how many videos we have left.

In [ ]:
# Check the length of the DataFrame
len(df)

312

We also need to remove all the "Wake Up" shows, special epsidodes with a co-host.

In [ ]:
# Fide all the "Wake Up" shows (case-insensitive)
wake_df = df[df['title'].str.contains(r'\bwake\b', case=False, na=False, regex=True)]

# Display and count them
print(f"Found {len(wake_df)} 'Wake' shows.")
wake_df[['title', 'duration_seconds', 'views_lifetime']]

Found 14 'Wake' shows.


,title,duration_seconds,views_lifetime
33,McCaffrey's Night Lifts 49ers Over Falcons - W...,6105.0,4152.0
35,49ers Super Bowl Hopes ROCKED by Fred Warner’s...,6792.0,3909.0
36,Purdy & Bosa Get Out Of Seattle With A Win - W...,5033.0,3873.0
37,49ers Were Their Worst Selves In Jax Loss - Wa...,5879.0,3830.0
55,49ers Wake Up. Jake Moody the Hero?,6302.0,3004.0
56,"49ers Wake Up: Moody Miss MORE WR Drama, Key R...",5011.0,3004.0
74,49ers Lose To Broncos In Preseason Game #1 | W...,3485.0,2600.0
101,"49ers Beat Giants, Mykel ACL (Of Course) WAKE ...",5944.0,2214.0
108,"49ers Helpless In Houston - Wake Up, with Damo...",5877.0,2122.0
110,What to Watch: 49ers vs Raiders Preseason Clas...,5908.0,2108.0


Let's get rid of those.

In [ ]:
# Drop all remaining "Wake Up" shows
df = df[~df['title'].str.contains(r'\bwake\b', case=False, na=False, regex=True)].copy()

# Confirm final count
print(f"Remaining videos after dropping Wake Up shows: {len(df)}")

Remaining videos after dropping Wake Up shows: 298


After filtiering out the livestreams and Wake Up shows, we're left with a clean dataset of **298 daily shows** -- each representing one consistent product and format. This makes sure that the model is learning from comparable examples, not getting confused by long-form watch parties.

And now to do some feature engineering.

## Engineer new input features

This is a *daily* YouTube show, essentially, a podcast. Day of the week -- and month of the year -- are potentially important features that the model can learn from. Let's build a couple of new features.

In [ ]:
# Create day of week (0 = Monday, 6 = Sunday)
df['day_of_week'] = df['publish_date'].dt.weekday

# Create month (0 = Jan, 11 = Dec)
df['month'] = df['publish_date'].dt.month - 1

# A quick check
df.head()

,video_id,title,publish_date,duration_seconds,views_lifetime,subs_lifetime,day_of_week,month
3,TUbMw5E7WWQ,Radio Layoffs Hit KNBR (Again) - Damon & Sam B...,2024-11-14,5306.0,10666.0,45.0,3,10
16,S1X12mROnSs,Analysis: 49ers Lose To Lions In Disastrous Fa...,2024-12-31,5234.0,5617.0,5.0,1,11
19,4BWOPFouLSU,"49ers Insider Matt Maiocco Talks Trades, QB Co...",2025-10-07,4611.0,5246.0,13.0,1,9
21,g2olKzhJ6JQ,"49ers Analysis: Campbell Is A LOSER, Purdy TER...",2024-12-13,5875.0,4995.0,9.0,4,11
23,m4t2Who5AeQ,49ers Analysis: Embarrassing Late-Game Loss vs...,2024-12-23,7007.0,4814.0,37.0,0,11


As a final step, let's streamline this dataset for modeling. We can remove `video_id`, which provides no predictive power and also `publish_date`, which has been made redundant by the featured we just engineered.

In [ ]:
# Keep only the relevant features for modeling
df = df[['title', 'duration_seconds', 'day_of_week', 'month', 'views_lifetime', 'subs_lifetime']]

# Check the results
df.head()

,title,duration_seconds,day_of_week,month,views_lifetime,subs_lifetime
3,Radio Layoffs Hit KNBR (Again) - Damon & Sam B...,5306.0,3,10,10666.0,45.0
16,Analysis: 49ers Lose To Lions In Disastrous Fa...,5234.0,1,11,5617.0,5.0
19,"49ers Insider Matt Maiocco Talks Trades, QB Co...",4611.0,1,9,5246.0,13.0
21,"49ers Analysis: Campbell Is A LOSER, Purdy TER...",5875.0,4,11,4995.0,9.0
23,49ers Analysis: Embarrassing Late-Game Loss vs...,7007.0,0,11,4814.0,37.0


This version of the dataset is now model-ready -- compact, consistent, and representative of the flagship daily show. Now, we'll save it for future use.

In [ ]:
# Save the cleaned and engineered dataset with .to_csv and .to_pickle
df.to_csv('/content/drive/MyDrive/Colab Notebooks/project_delphi/data/project_delphi_clean.csv', index=False)
df.to_pickle('/content/drive/MyDrive/Colab Notebooks/project_delphi/data/project_delphi_clean.pkl')